# CardioCCC NER Pipeline — lcampillos/roberta-es-clinical-trials-ner

This notebook preserves the original DisTEMIST training design and evaluates the ten-model ensemble on CardioCCC test.
The runtime configuration was chosen using short timing measurements on DisTEMIST training data only; CardioCCC results were not used to choose it.

## Contents

1. [Environment and dependencies](#1-environment-and-dependencies)
2. [Configuration](#2-configuration)
   - [Paths and dataset](#21-paths-and-dataset)
   - [Labels and data loading](#22-labels-and-data-loading)
   - [Segmentation and label alignment](#23-segmentation-and-label-alignment)
   - [Hyperparameters and tokenizer](#24-hyperparameters-and-tokenizer)
3. [Training](#3-training)
   - [Evaluation metrics](#31-evaluation-metrics)
   - [Discriminative fine-tuning](#32-discriminative-fine-tuning)
   - [Multi-seed k-fold loop](#33-multi-seed-k-fold-loop)
   - [Ensemble summary](#34-ensemble-summary)
4. [Inference](#4-inference)
   - [Sentence-based inference function](#41-sentence-based-inference-function)
   - [Ensemble execution](#42-ensemble-execution)
5. [Evaluation](#5-evaluation)
   - [Strict offset-based evaluation](#51-strict-offset-based-evaluation)
   - [Overlap evaluation (IoU)](#52-overlap-evaluation-iou)

## 1. Environment and dependencies

Installation of the required packages and import of libraries.

In [1]:
%pip install -q "evaluate==0.4.6" "spacy==3.8.7" "datasets==3.6.0" "transformers==4.57.6" "accelerate==1.12.0"
%pip install -q --use-pep517 "seqeval==1.2.2"
%pip install -q "https://github.com/explosion/spacy-models/releases/download/es_core_news_md-3.8.0/es_core_news_md-3.8.0-py3-none-any.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.4/33.4 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import re
import time
from collections import defaultdict
from pathlib import Path

import evaluate
import numpy as np
import pandas as pd
import spacy
import torch
from torch.optim import AdamW
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
from transformers import TrainerCallback
import importlib.metadata
import math

if torch.cuda.device_count() != 2:
    raise RuntimeError("Select GPU T4 x2: this execution expects two GPUs.")
print(f"GPU count: {torch.cuda.device_count()} | CPU threads: {torch.get_num_threads()}")


GPU available: True
GPU: Tesla T4
GPU count: 2 | CPU threads: 2


## 2. Configuration

### 2.1. Paths and dataset

Definition of dataset paths and selection of the base model. Training uses the 750 DisTEMIST training documents; external evaluation uses the 250 CardioCCC test documents.

In [3]:
import csv
import hashlib
from zipfile import ZipFile

INPUT_ROOT = Path("/kaggle/input")
DATASET_ROOT = Path("/kaggle/working/cardioccc_dataset")
EXPECTED_FINGERPRINT = "3cef134a65040d6d1b12b3594cd30dab127a24ceda13187159de9429ad947965"
dataset_dirs = set()
for path in INPUT_ROOT.rglob("dataset_manifest.json"):
    try:
        candidate = json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, UnicodeDecodeError):
        continue
    if candidate.get("dataset_id") == "distemist-cardioccc-external-v1":
        dataset_dirs.add(path.parent.resolve())

if len(dataset_dirs) == 1:
    DATASET_ROOT = dataset_dirs.pop()
elif dataset_dirs:
    raise RuntimeError("Multiple copies of the prepared dataset were found in the inputs.")
else:
    archives = sorted(INPUT_ROOT.rglob("distemist_cardioccc_datos.zip"))
    if len(archives) != 1:
        raise RuntimeError("Attach the distemist-cardioccc-external-evaluation dataset, extracted or as a ZIP archive.")
    with ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            if Path(name).is_absolute() or ".." in Path(name).parts:
                raise ValueError(f"Unsafe archive path: {name}")
        archive.extractall(DATASET_ROOT)

manifest = json.loads((DATASET_ROOT / "dataset_manifest.json").read_text(encoding="utf-8"))
fingerprint = hashlib.sha256(json.dumps(manifest["files"], sort_keys=True, separators=(",", ":")).encode()).hexdigest()
assert fingerprint == manifest["fingerprint"] == EXPECTED_FINGERPRINT
for name, digest in manifest["files"].items():
    assert hashlib.sha256((DATASET_ROOT / name).read_bytes()).hexdigest() == digest, name
print(f"Dataset verified: {DATASET_ROOT}")

train_jsonl = DATASET_ROOT / "distemist/distemist_train.jsonl"
assert hashlib.sha256(train_jsonl.read_bytes()).hexdigest() == "f0200564e73f8665949d09ba32d055c7435dde794f47d950d43141894cf19471"
test_dir = DATASET_ROOT / "cardioccc/test/texts"
texts = {p.stem: p.read_bytes().decode("utf-8") for p in sorted(test_dir.glob("*.txt"))}
assert len(texts) == 250

with (DATASET_ROOT / "cardioccc/test/gold.tsv").open(encoding="utf-8", newline="") as stream:
    gold = list(csv.DictReader(stream, delimiter="\t"))
assert len(gold) == 7884 and {row["filename"] for row in gold} == set(texts)
for row in gold:
    start, end = int(row["start_span"]), int(row["end_span"])
    assert row["label"] == "ENFERMEDAD" and 0 <= start < end <= len(texts[row["filename"]])
    assert texts[row["filename"]][start:end] == row["text"]

gs_path = Path("/kaggle/working/cardioccc_test_mentions.tsv")
with gs_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.writer(stream, delimiter="\t")
    writer.writerow(["filename", "label", "off0", "off1", "span"])
    writer.writerows((row["filename"], row["label"], row["start_span"], row["end_span"], row["text"]) for row in gold)

DATA_PATHS = {
    "train_jsonl": str(train_jsonl),
    "text_files_test_dir": str(test_dir),
    "gs_mentions_tsv": str(gs_path),
}
BASE_MODEL = "lcampillos/roberta-es-clinical-trials-ner"

print("Configured paths:")
for k, v in DATA_PATHS.items():
    print(f"  - {k}: {v}")
print(f"Base model: {BASE_MODEL}")
BASE_MODEL_REVISION = 'cfdd0455be8bd539a27883026b24aa745c871493'


Dataset verified: /kaggle/input/distemist-cardioccc-external-evaluation
Configured paths:
  - train_jsonl: /kaggle/input/distemist-cardioccc-external-evaluation/distemist/distemist_train.jsonl
  - text_files_test_dir: /kaggle/input/distemist-cardioccc-external-evaluation/cardioccc/test/texts
  - gs_mentions_tsv: /kaggle/working/cardioccc_test_mentions.tsv
Base model: lcampillos/roberta-es-clinical-trials-ner


### 2.2. Labels and data loading

BIO label mapping and loading of the training JSONL file.

In [4]:
id2label = {0: "B-ENFERMEDAD", 1: "I-ENFERMEDAD", 2: "O"}
label2id = {"B-ENFERMEDAD": 0, "I-ENFERMEDAD": 1, "O": 2}
label_list = [id2label[i] for i in range(len(id2label))]

nlp_spacy = spacy.load("es_core_news_md")

from datasets import load_dataset as _load_dataset
train_full = _load_dataset("json", data_files=DATA_PATHS["train_jsonl"], split="train")

print(f"Labels: {label2id}")
print(f"Training documents: {len(train_full)}")

Generating train split: 0 examples [00:00, ? examples/s]

Labels: {'B-ENFERMEDAD': 0, 'I-ENFERMEDAD': 1, 'O': 2}
Training documents: 750


### 2.3. Segmentation and label alignment

Sentence segmentation functions using spaCy and BIO label alignment during tokenization.

In [5]:
def split_by_sentences(text, tokens, labels, nlp_spacy):
    """Split document tokens and labels into sentences using spaCy."""
    doc = nlp_spacy(text)
    sentences = list(doc.sents)

    if len(sentences) <= 1:
        return [(tokens, labels)]

    token_char_starts = []
    search_pos = 0
    for tok in tokens:
        idx = text.find(tok, search_pos)
        if idx == -1:
            return [(tokens, labels)]
        token_char_starts.append(idx)
        search_pos = idx + len(tok)

    results = []
    for sent in sentences:
        sent_start = sent.start_char
        sent_end = sent.end_char
        sent_token_indices = [
            i for i, cs in enumerate(token_char_starts)
            if sent_start <= cs < sent_end
        ]
        if not sent_token_indices:
            continue
        sent_tokens = [tokens[i] for i in sent_token_indices]
        sent_labels = [labels[i] for i in sent_token_indices]
        results.append((sent_tokens, sent_labels))

    return results if results else [(tokens, labels)]


def tokenize_and_align_labels(examples, tok, nlp_spacy, max_length=512):
    """Tokenize sentences with truncation=True and propagate B->I to subtokens."""
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for doc_idx in range(len(examples["tokens"])):
        text = examples["text"][doc_idx]
        tokens = examples["tokens"][doc_idx]
        ner_tags = examples["ner_tags"][doc_idx]

        sent_chunks = split_by_sentences(text, tokens, ner_tags, nlp_spacy)

        for sent_tokens, sent_labels in sent_chunks:
            tokenized = tok(
                [sent_tokens],
                is_split_into_words=True,
                truncation=True,
                max_length=max_length,
                padding=False,
            )

            word_ids = tokenized.word_ids(batch_index=0)
            previous_word_idx = None
            label_ids = []

            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(sent_labels[word_idx])
                else:
                    prev_label = sent_labels[word_idx]
                    label_ids.append(1 if prev_label == 0 else prev_label)
                previous_word_idx = word_idx

            all_input_ids.append(tokenized["input_ids"][0])
            all_attention_masks.append(tokenized["attention_mask"][0])
            all_labels.append(label_ids)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    }

### 2.4. Hyperparameters and tokenizer

Experiment configuration: training hyperparameters and tokenizer loading.

In [6]:
BASE_MODEL_TAG = BASE_MODEL.split("/")[-1]
USE_BF16 = True
USE_FP16 = False
CHECKPOINT_REENTRANT = False
CPU_THREADS = 2
torch.set_num_threads(CPU_THREADS)

MAX_EPOCHS          = 20
BATCH_SIZE          = 16
LEARNING_RATE       = 8.516e-5
LR_LAYER_DECAY      = 0.95
LR_ENCODER_GROUPS   = 3
DROPOUT             = 0.1
WEIGHT_DECAY        = 0.1844
WARMUP_RATIO        = 0.1
EARLY_STOPPING_PATIENCE   = 5
EARLY_STOPPING_THRESHOLD  = 1e-4

K_FOLDS             = 5
CV_SPLIT_SEED       = 42
SEEDS               = [123, 4242]
ENSEMBLE_VOTING_RATIO = 0.5

RESULTS_DIR        = f"results_{BASE_MODEL_TAG}_kfold_multiseed"
MODEL_OUTPUT_PREFIX = f"{BASE_MODEL_TAG}-distemist-ner"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

config = AutoConfig.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT,
    classifier_dropout=DROPOUT,
    attn_implementation="sdpa",
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION,
    add_prefix_space=True,
    do_lower_case=False,
    keep_accents=True,
    model_max_length=config.max_position_embeddings,
)

hyperparams = {
    "base_model": BASE_MODEL, "base_model_tag": BASE_MODEL_TAG,
    "max_epochs": MAX_EPOCHS, "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "lr_layer_decay": LR_LAYER_DECAY,
    "lr_encoder_groups": LR_ENCODER_GROUPS, "dropout": DROPOUT,
    "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    "k_folds": K_FOLDS, "cv_split_seed": CV_SPLIT_SEED,
    "seeds": SEEDS, "ensemble_voting_ratio": ENSEMBLE_VOTING_RATIO,
}
with open(f"{RESULTS_DIR}/hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(hyperparams, f, ensure_ascii=False, indent=2)

print(f"Model: {BASE_MODEL} | Max pos embeddings: {config.max_position_embeddings}")
print(f"Results directory: {RESULTS_DIR}")
runtime_settings = {
    "model_revision": BASE_MODEL_REVISION,
    "bf16": USE_BF16, "fp16": USE_FP16,
    "gradient_checkpointing_use_reentrant": CHECKPOINT_REENTRANT,
    "cpu_threads": CPU_THREADS, "gpu_count": torch.cuda.device_count(),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "software": {p: importlib.metadata.version(p) for p in
                 ["torch", "transformers", "accelerate", "datasets", "spacy", "seqeval"]},
    "cuda": torch.version.cuda,
    "training_corpus": "DisTEMIST train (750 documents)",
    "external_test": "CardioCCC test (250 documents)",
    "runtime_selection": "Throughput and finite loss on DisTEMIST train only",
}
runtime_settings["speed_benchmark"] = {'steps_per_second': 2.0366128105347236, 'observed_query_dtypes': {'cuda:1': {'dtype': 'torch.float32', 'autocast': False}, 'cuda:0': {'dtype': 'torch.float32', 'autocast': False}}, 'measured_on': 'DisTEMIST training fold 1; 56 timed optimizer steps'}
with open(f"{RESULTS_DIR}/runtime_settings.json", "w", encoding="utf-8") as stream:
    json.dump(runtime_settings, stream, indent=2)
print("RUNTIME " + json.dumps(runtime_settings), flush=True)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Model: lcampillos/roberta-es-clinical-trials-ner | Max pos embeddings: 514
Results directory: results_roberta-es-clinical-trials-ner_kfold_multiseed
RUNTIME {"model_revision": "cfdd0455be8bd539a27883026b24aa745c871493", "bf16": true, "fp16": false, "gradient_checkpointing_use_reentrant": false, "cpu_threads": 2, "gpu_count": 2, "gpu_names": ["Tesla T4", "Tesla T4"], "software": {"torch": "2.11.0+cu128", "transformers": "4.57.6", "accelerate": "1.12.0", "datasets": "3.6.0", "spacy": "3.8.7", "seqeval": "1.2.2"}, "cuda": "12.8", "training_corpus": "DisTEMIST train (750 documents)", "external_test": "CardioCCC test (250 documents)", "runtime_selection": "Throughput and finite loss on DisTEMIST train only", "speed_benchmark": {"steps_per_second": 2.0366128105347236, "observed_query_dtypes": {"cuda:1": {"dtype": "torch.float32", "autocast": false}, "cuda:0": {"dtype": "torch.float32", "autocast": false}}, "measured_on": "DisTEMIST training fold 1; 56 timed optimizer steps"}}


## 3. Training

### 3.1. Evaluation metrics

Definition of the seqeval metrics for NER evaluation during training.

In [7]:
metric_fn = evaluate.load("seqeval", trust_remote_code=True)


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[pred] for (pred, la) in zip(prediction, label) if la != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[la] for (_, la) in zip(prediction, label) if la != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric_fn.compute(
        predictions=true_predictions,
        references=true_labels,
        zero_division=0.0,
    )
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

### 3.2. Discriminative fine-tuning

Assignment of different learning rates according to layer depth.

In [8]:
def create_discriminative_optimizer(model):
    named_params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    layer_re = re.compile(r"\.(?:encoder\.layer|layer|layers|block|h)\.(\d+)\.")
    layer_ids = [int(m.group(1)) for n, _ in named_params for m in [layer_re.search(n.lower())] if m]
    max_layer_id = max(layer_ids) if layer_ids else 0

    groups = {}
    for name, param in named_params:
        lname = name.lower()
        if "classifier" in lname or "crf" in lname:
            bucket, lr = "head", LEARNING_RATE
        elif "embed" in lname:
            bucket, lr = "embeddings", LEARNING_RATE * (LR_LAYER_DECAY ** (LR_ENCODER_GROUPS + 1))
        else:
            match = layer_re.search(lname)
            if match and max_layer_id > 0:
                zone = min(int((int(match.group(1)) / max_layer_id) * LR_ENCODER_GROUPS), LR_ENCODER_GROUPS - 1)
                bucket, lr = f"encoder_{zone}", LEARNING_RATE * (LR_LAYER_DECAY ** (LR_ENCODER_GROUPS - zone))
            else:
                bucket, lr = "head", LEARNING_RATE

        if bucket not in groups:
            groups[bucket] = {"params": [], "lr": float(lr), "param_count": 0, "tensor_count": 0}
        groups[bucket]["params"].append(param)
        groups[bucket]["param_count"] += param.numel()
        groups[bucket]["tensor_count"] += 1

    optimizer = AdamW(
        [{"params": g["params"], "lr": g["lr"], "weight_decay": WEIGHT_DECAY} for g in groups.values()],
        lr=LEARNING_RATE,
        fused=torch.cuda.is_available(),
    )
    order = ["embeddings"] + [f"encoder_{i}" for i in range(LR_ENCODER_GROUPS)] + ["head"]
    summary = [
        {"bucket": b, "lr": groups[b]["lr"], "param_count": groups[b]["param_count"], "tensor_count": groups[b]["tensor_count"]}
        for b in order if b in groups
    ]
    return optimizer, summary

### 3.3. Multi-seed k-fold loop

Training across document folds and seeds with early stopping on internal validation. The resulting models form the ensemble. CardioCCC is reserved for external evaluation.

In [9]:
class TrainingProgress(TrainerCallback):
    def __init__(self, fold, seed):
        self.fold = fold
        self.seed = seed
        self.started = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.started = time.perf_counter()
        print(f"TRAIN_START fold={self.fold} seed={self.seed} GPUs={args.n_gpu} effective_batch={args.train_batch_size}", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        elapsed = time.perf_counter() - self.started
        for key in ["loss", "eval_loss"]:
            if key in (logs or {}) and not math.isfinite(logs[key]):
                raise FloatingPointError(f"Non-finite {key}: {logs[key]}")
        report = {"fold": self.fold, "seed": self.seed,
                  "epoch": round(state.epoch or 0, 3), "step": state.global_step,
                  "elapsed_minutes": round(elapsed / 60, 2),
                  "average_steps_per_second": round(state.global_step / elapsed, 3),
                  **(logs or {})}
        print("TRAIN_PROGRESS " + json.dumps(report), flush=True)

    def on_epoch_end(self, args, state, control, **kwargs):
        self.epoch_train_end = time.perf_counter()
        print(f"EPOCH_TRAIN_END fold={self.fold} seed={self.seed} epoch={state.epoch:.2f}", flush=True)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        self.evaluation_end = time.perf_counter()
        seconds = self.evaluation_end - self.epoch_train_end
        print(f"VALIDATION_TIME fold={self.fold} seed={self.seed} seconds={seconds:.2f}", flush=True)

    def on_save(self, args, state, control, **kwargs):
        seconds = time.perf_counter() - self.evaluation_end
        print(f"CHECKPOINT_TIME fold={self.fold} seed={self.seed} seconds={seconds:.2f}", flush=True)


def make_kfold_indices(n_samples, k_folds, split_seed):
    rng = np.random.default_rng(split_seed)
    indices = np.arange(n_samples)
    rng.shuffle(indices)
    fold_sizes = np.full(k_folds, n_samples // k_folds, dtype=int)
    fold_sizes[: n_samples % k_folds] += 1
    folds, current = [], 0
    for size in fold_sizes:
        val_idx = indices[current: current + size]
        train_idx = np.concatenate((indices[:current], indices[current + size:]))
        folds.append((train_idx, val_idx))
        current += size
    return folds


fold_seed_results = []
ensemble_models = []
folds = make_kfold_indices(len(train_full), K_FOLDS, CV_SPLIT_SEED)

print(f"Multi-seed k-fold training | docs={len(train_full)} | folds={K_FOLDS} | seeds={SEEDS}")

for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):
    train_raw = train_full.select(train_idx.tolist())
    val_raw   = train_full.select(val_idx.tolist())

    map_kwargs = dict(batched=True, remove_columns=train_full.column_names)
    fn = lambda x: tokenize_and_align_labels(x, tokenizer, nlp_spacy, max_length=512)
    train_ds = train_raw.map(fn, **map_kwargs)
    val_ds   = val_raw.map(fn, **map_kwargs)

    print(f"\nFold {fold_idx}/{K_FOLDS} | train={len(train_raw)} docs / {len(train_ds)} seqs | val={len(val_raw)} docs / {len(val_ds)} seqs")

    for seed in SEEDS:
        print(f"  Seed {seed}...")
        set_seed(seed)
        start_time = time.time()

        model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION, config=config,ignore_mismatched_sizes=True)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": CHECKPOINT_REENTRANT})
        optimizer, lr_summary = create_discriminative_optimizer(model)

        output_dir = f"{RESULTS_DIR}/{MODEL_OUTPUT_PREFIX}-fold{fold_idx}-seed{seed}"
        training_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            logging_strategy="steps",
            logging_steps=100,
            logging_nan_inf_filter=False,
            save_strategy="epoch",
            num_train_epochs=MAX_EPOCHS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1",
            greater_is_better=True,
            save_total_limit=1,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=BATCH_SIZE,
            dataloader_num_workers=2,
            dataloader_prefetch_factor=4,
            dataloader_persistent_workers=True,
            seed=seed,
            bf16=USE_BF16,
            fp16=USE_FP16,
            save_only_model=True,
            report_to="none",
        )

        trainer = Trainer(
            model, training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
            data_collator=DataCollatorForTokenClassification(tokenizer),
            callbacks=[TrainingProgress(fold_idx, seed), EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
            )],
            optimizers=(optimizer, None),
        )
        trainer.train()

        model_dir = trainer.state.best_model_checkpoint or output_dir
        val_metrics = trainer.evaluate(val_ds)
        best_logs = [l for l in trainer.state.log_history if "eval_f1" in l]
        best_f1   = max((l["eval_f1"] for l in best_logs), default=float("nan"))
        elapsed   = (time.time() - start_time) / 60

        row = {
            "fold": fold_idx, "seed": seed,
            "train_docs": len(train_raw), "val_docs": len(val_raw),
            "train_sequences": len(train_ds), "val_sequences": len(val_ds),
            "best_eval_f1": best_f1,
            "eval_precision": val_metrics.get("eval_precision", float("nan")),
            "eval_recall":    val_metrics.get("eval_recall",    float("nan")),
            "eval_f1":        val_metrics.get("eval_f1",        float("nan")),
            "eval_accuracy":  val_metrics.get("eval_accuracy",  float("nan")),
            "eval_loss":      val_metrics.get("eval_loss",      float("nan")),
            "elapsed_min": elapsed, "model_dir": model_dir,
        }
        fold_seed_results.append(row)
        ensemble_models.append({"fold": fold_idx, "seed": seed, "model_dir": model_dir, "eval_f1": row["eval_f1"]})

        print(f"    fold={fold_idx} seed={seed} | best_f1={best_f1:.4f} | eval_f1={row['eval_f1']:.4f} | {elapsed:.1f} min")

        del trainer, model, optimizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if not fold_seed_results:
    raise RuntimeError("No fold-seed model was trained.")

df_ensemble_results = (
    pd.DataFrame(fold_seed_results)
    .sort_values(["eval_f1", "fold", "seed"], ascending=[False, True, True])
    .reset_index(drop=True)
)
df_ensemble_results.to_csv(f"{RESULTS_DIR}/ensemble_fold_seed_summary.csv", index=False)
with open(f"{RESULTS_DIR}/ensemble_fold_seed_summary.json", "w", encoding="utf-8") as f:
    json.dump(fold_seed_results, f, ensure_ascii=False, indent=2)

print(f"\nEnsemble models: {len(ensemble_models)}")
print(df_ensemble_results[["fold","seed","eval_f1","best_eval_f1","elapsed_min"]].to_string(index=False))

Multi-seed k-fold training | docs=750 | folds=5 | seeds=[123, 4242]


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


Fold 1/5 | train=600 docs / 9485 seqs | val=150 docs / 2228 seqs
  Seed 123...


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

...

  Seed 4242...
TRAIN_START fold=5 seed=4242 GPUs=2 effective_batch=32


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.115400,0.095220,0.639418,0.752306,0.691283,0.966079
2,0.085100,0.107356,0.704359,0.734519,0.719123,0.969358
3,0.051400,0.090749,0.677303,0.741107,0.707770,0.966844
4,0.030100,0.097427,0.691071,0.764822,0.726079,0.966931
5,0.018300,0.125312,0.717762,0.777339,0.746363,0.970167
6,0.011000,0.125916,0.730696,0.754282,0.742301,0.970109
7,0.008100,0.143053,0.700119,0.772069,0.734336,0.969257
8,0.006600,0.148574,0.686705,0.782609,0.731527,0.967827
9,0.003500,0.165548,0.725994,0.781950,0.752934,0.969401
10,0.003900,0.172611,0.732176,0.777997,0.754392,0.969777


TRAIN_PROGRESS {"fold": 5, "seed": 4242, "epoch": 0.3401360544217687, "step": 100, "elapsed_minutes": 0.77, "average_steps_per_second": 2.154, "loss": 0.4708, "grad_norm": 1.5513705015182495, "learning_rate": 1.1678523593112244e-05}
TRAIN_PROGRESS {"fold": 5, "seed": 4242, "epoch": 0.6802721088435374, "step": 200, "elapsed_minutes": 1.54, "average_steps_per_second": 2.166, "loss": 0.1154, "grad_norm": 1.2336220741271973, "learning_rate": 2.3475012071003398e-05}
EPOCH_TRAIN_END fold=5 seed=4242 epoch=1.00
TRAIN_PROGRESS {"fold": 5, "seed": 4242, "epoch": 1.0, "step": 294, "elapsed_minutes": 2.45, "average_steps_per_second": 2.0, "eval_loss": 0.09521998465061188, "eval_precision": 0.6394176931690929, "eval_recall": 0.7523056653491436, "eval_f1": 0.6912832929782082, "eval_accuracy": 0.9660786778196738, "eval_runtime": 12.6733, "eval_samples_per_second": 182.668, "eval_steps_per_second": 11.441}
VALIDATION_TIME fold=5 seed=4242 seconds=12.68
CHECKPOINT_TIME fold=5 seed=4242 seconds=0.72
TR

TRAIN_PROGRESS {"fold": 5, "seed": 4242, "epoch": 20.0, "step": 5880, "elapsed_minutes": 49.35, "average_steps_per_second": 1.986, "eval_loss": 0.22537723183631897, "eval_precision": 0.7454772301933874, "eval_recall": 0.7872200263504612, "eval_f1": 0.7657801986542774, "eval_accuracy": 0.9713951371733195, "eval_runtime": 12.5577, "eval_samples_per_second": 184.349, "eval_steps_per_second": 11.547}
VALIDATION_TIME fold=5 seed=4242 seconds=25.95
    fold=5 seed=4242 | best_f1=0.7658 | eval_f1=0.7658 | 49.4 min

Ensemble models: 10
 fold  seed  eval_f1  best_eval_f1  elapsed_min
    4   123 0.784204      0.784204    49.369440
    4  4242 0.778331      0.778331    48.938749
    1  4242 0.774704      0.774704    49.721082
    1   123 0.771883      0.771883    42.315370
    2   123 0.770846      0.770846    41.673074
    5  4242 0.765780      0.765780    49.366597
    3  4242 0.764253      0.764253    49.898335
    5   123 0.763413      0.763413    49.423240
    2  4242 0.759632      0.759632

### 3.4. Ensemble summary

Aggregation of validation metrics across folds and seeds, and saving of the ensemble state.

In [10]:
cols = ["eval_precision", "eval_recall", "eval_f1", "eval_accuracy", "eval_loss"]
agg = df_ensemble_results[cols].agg(["mean", "std", "min", "max"]).T.reset_index().rename(columns={"index": "metric"})
print(agg.to_string(index=False))

summary = {c: {"mean": float(df_ensemble_results[c].mean()), "std": float(df_ensemble_results[c].std(ddof=0))} for c in cols}
summary["ensemble_size"] = len(ensemble_models)
with open(f"{RESULTS_DIR}/validation_ensemble_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

        metric     mean      std      min      max
eval_precision 0.758439 0.014338 0.738814 0.781630
   eval_recall 0.778982 0.012295 0.755166 0.791588
       eval_f1 0.768468 0.009505 0.751636 0.784204
 eval_accuracy 0.971978 0.001760 0.969986 0.974739
     eval_loss 0.184933 0.034554 0.120669 0.225377


## 4. Inference

### 4.1. Sentence-based inference function

Segment each document using spaCy, apply the NER pipeline to each sentence and adjust offsets to the full text.

In [11]:
def sentence_based_ner(texto, pipeline_ner, nlp_spacy):
    """Run sentence-based NER and map offsets to the full document."""
    doc = nlp_spacy(texto)
    all_entities = []

    for sent in doc.sents:
        sent_text = sent.text
        sent_offset = sent.start_char

        entities = pipeline_ner(sent_text)

        for entity in entities:
            entity["start"] += sent_offset
            entity["end"] += sent_offset
            all_entities.append(entity)

    return all_entities

### 4.2. Ensemble execution

Read the test texts, run inference with each ensemble model and aggregate entities by majority voting.

In [12]:
ruta_txts = DATA_PATHS["text_files_test_dir"]
ruta_gs   = DATA_PATHS["gs_mentions_tsv"]

texts_by_filename = {
    f.replace(".txt", ""): open(os.path.join(ruta_txts, f), encoding="utf-8").read()
    for f in sorted(os.listdir(ruta_txts)) if f.endswith(".txt")
}

if not texts_by_filename:
    raise RuntimeError(f"No .txt files were found in {ruta_txts}")
if not ensemble_models:
    raise RuntimeError("The ensemble contains no models.")

vote_threshold = max(1, int(np.ceil(ENSEMBLE_VOTING_RATIO * len(ensemble_models))))
pred_file = f"{RESULTS_DIR}/predictions_ensemble_k{K_FOLDS}_s{len(SEEDS)}.tsv"
aggregated = defaultdict(int)
model_times = []
end_to_end_start = t0 = time.time()

print(f"Test documents: {len(texts_by_filename)} | Models: {len(ensemble_models)} | Required votes: {vote_threshold}")

for model_info in ensemble_models:
    fold, seed, model_dir = model_info["fold"], model_info["seed"], model_info["model_dir"]
    t_model = time.time()

    modelo_inf    = AutoModelForTokenClassification.from_pretrained(model_dir)
    tokenizer_inf = AutoTokenizer.from_pretrained(model_dir)
    nlp_ner = pipeline("ner", model=modelo_inf, tokenizer=tokenizer_inf, aggregation_strategy="simple")

    for filename, texto in texts_by_filename.items():
        for ent in sentence_based_ner(texto, nlp_ner, nlp_spacy):
            if ent["entity_group"] == "ENFERMEDAD":
                aggregated[(filename, int(ent["start"]), int(ent["end"]))] += 1

    elapsed = time.time() - t_model
    model_times.append(elapsed)
    print(f"  fold={fold} seed={seed}: {elapsed:.1f}s")

    del nlp_ner, tokenizer_inf, modelo_inf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Voting consensus
mark_counter = defaultdict(int)
final_rows = []
for (filename, off0, off1), votes in sorted(aggregated.items()):
    if votes < vote_threshold:
        continue
    mark_counter[filename] += 1
    final_rows.append({
        "filename": filename,
        "mark": f"T{mark_counter[filename]}",
        "label": "ENFERMEDAD",
        "off0": off0, "off1": off1,
        "span": texts_by_filename[filename][off0:off1],
    })

df_pred = pd.DataFrame(final_rows, columns=["filename", "mark", "label", "off0", "off1", "span"])
df_pred.to_csv(pred_file, sep="\t", index=False)

total_s = time.time() - t0
stats = {
    "archivos_procesados": len(texts_by_filename),
    "modelos_ensamblados": len(ensemble_models),
    "voting_ratio": ENSEMBLE_VOTING_RATIO,
    "votos_requeridos": vote_threshold,
    "entidades_candidatas": len(aggregated),
    "entidades_detectadas": len(df_pred),
    "inference_total_seconds": total_s,
    "inference_avg_file_seconds": total_s / len(texts_by_filename),
    "inference_avg_model_seconds": float(np.mean(model_times)),
}
with open(f"{RESULTS_DIR}/inference_stats_ensemble.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"Detected entities: {len(df_pred)} | Total time: {total_s:.1f}s | Predictions: {pred_file}")

Test documents: 250 | Models: 10 | Required votes: 5


Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  fold=1 seed=123: 172.6s


Device set to use cuda:0


  fold=1 seed=4242: 167.9s


...

  fold=4 seed=4242: 168.7s


Device set to use cuda:0


  fold=5 seed=123: 170.9s


Device set to use cuda:0


  fold=5 seed=4242: 172.7s
Detected entities: 7747 | Total time: 1707.7s | Predictions: results_roberta-es-clinical-trials-ner_kfold_multiseed/predictions_ensemble_k5_s2.tsv


## 5. Evaluation

### 5.1. Strict offset-based evaluation

Compare predictions against the gold standard using exact label and character-offset matches across all 250 documents.

In [13]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f

df_gs   = pd.read_csv(ruta_gs,   sep="\t")
df_pred = pd.read_csv(pred_file, sep="\t")

set_gs   = set(zip(df_gs["filename"],   df_gs["label"],   df_gs["off0"],   df_gs["off1"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["off0"], df_pred["off1"]))

tp, fp, fn = len(set_gs & set_pred), len(set_pred - set_gs), len(set_gs - set_pred)
precision, recall, fscore = prf(tp, fp, fn)
end_to_end_seconds = time.time() - end_to_end_start

strict_report = {
    "base_model": BASE_MODEL, "k_folds": K_FOLDS, "seeds": SEEDS,
    "ensemble_size": len(ensemble_models), "voting_ratio": ENSEMBLE_VOTING_RATIO,
    "tp": tp, "fp": fp, "fn": fn,
    "precision": precision, "recall": recall, "fscore": fscore,
    "end_to_end_seconds": end_to_end_seconds,
    "predictions_file": pred_file,
}
with open(f"{RESULTS_DIR}/strict_evaluation_ensemble.json", "w", encoding="utf-8") as f:
    json.dump(strict_report, f, ensure_ascii=False, indent=2)

print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {fscore:.4f}")
print(f"TP={tp} FP={fp} FN={fn} | End-to-end: {end_to_end_seconds:.1f}s")

Precision: 0.7671 | Recall: 0.7538 | F1: 0.7604
TP=5943 FP=1804 FN=1941 | End-to-end: 1708.1s


### 5.2. Overlap evaluation (IoU)

Calculate precision, recall and F1 at different overlap thresholds between predicted and gold spans.

In [14]:
EVAL_SUMMARY_JSON = f"{RESULTS_DIR}/overlap_eval_summary.json"
processed_files   = sorted(texts_by_filename)
df_gs_filt        = df_gs

thresholds = [0.0, 0.5, 0.8]
results    = {t: {"tp": 0, "fp": 0, "fn": 0} for t in thresholds}

for filename in processed_files:
    gs_ints   = list(zip(df_gs_filt[df_gs_filt["filename"] == filename]["off0"],
                         df_gs_filt[df_gs_filt["filename"] == filename]["off1"]))
    pred_ints = list(zip(df_pred[df_pred["filename"] == filename]["off0"],
                         df_pred[df_pred["filename"] == filename]["off1"]))

    iou_matrix = sorted(
        [(max(0, min(p1,g1) - max(p0,g0)) / (max(p1,g1) - min(p0,g0)), pi, gi)
         for pi,(p0,p1) in enumerate(pred_ints)
         for gi,(g0,g1) in enumerate(gs_ints)
         if max(p1,g1) - min(p0,g0) > 0 and min(p1,g1) - max(p0,g0) > 0],
        reverse=True,
    )

    for t in thresholds:
        matched_p, matched_g = set(), set()
        for iou, pi, gi in iou_matrix:
            if iou >= t and pi not in matched_p and gi not in matched_g:
                matched_p.add(pi); matched_g.add(gi)
        tp = len(matched_p)
        results[t]["tp"] += tp
        results[t]["fp"] += len(pred_ints) - tp
        results[t]["fn"] += len(gs_ints)   - tp

report = {"Estricta": {**dict(zip(["tp","fp","fn"],[strict_report["tp"],strict_report["fp"],strict_report["fn"]])),
                       "precision": strict_report["precision"], "recall": strict_report["recall"], "fscore": strict_report["fscore"]}}
print(f"Strict: P={strict_report['precision']:.4f} R={strict_report['recall']:.4f} F1={strict_report['fscore']:.4f}\n")

for t in thresholds:
    tp, fp, fn = results[t]["tp"], results[t]["fp"], results[t]["fn"]
    p, r, f1 = prf(tp, fp, fn)
    report[f"IoU >= {t}"] = {"tp": tp, "fp": fp, "fn": fn, "precision": round(p,4), "recall": round(r,4), "fscore": round(f1,4)}
    print(f"IoU >= {t}: P={p:.4f} R={r:.4f} F1={f1:.4f} | TP={tp} FP={fp} FN={fn}")

with open(EVAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

Strict: P=0.7671 R=0.7538 F1=0.7604

IoU >= 0.0: P=0.8776 R=0.8624 F1=0.8699 | TP=6799 FP=948 FN=1085
IoU >= 0.5: P=0.8339 R=0.8194 F1=0.8266 | TP=6460 FP=1287 FN=1424
IoU >= 0.8: P=0.7782 R=0.7647 F1=0.7714 | TP=6029 FP=1718 FN=1855
